# Macros Avanzadas en Rust: Patrones y Casos de Uso Complejos

Las macros en Rust son una de las herramientas más potentes para reducir el código repetitivo (*boilerplate*) y crear abstracciones de alto nivel. Este notebook explora técnicas avanzadas que van más allá de simples sustituciones.

## 1. Macros Declarativas (`macro_rules!`) de Alto Nivel

### A. El Patrón de "Reglas Internas" (@)
Un patrón común en macros complejas es usar un prefijo especial (como `@`) para definir ramas que solo deben ser llamadas por la propia macro (recursión interna). Esto ayuda a modularizar la lógica de la macro.

In [ ]:
macro_rules! mi_json {
    // Punto de entrada para un objeto JSON
    ({ $($key:tt : $val:tt),* }) => {{
        let mut map = std::collections::HashMap::new();
        $(
            map.insert(stringify!($key), mi_json!(@valor $val));
        )*
        map
    }};

    // Regla interna para procesar valores
    (@valor { $($sub:tt)* }) => {
        // Si el valor es otro objeto, llamamos recursivamente
        serde_json::to_value(mi_json!({ $($sub)* })).unwrap()
    };
    
    (@valor $simple:expr) => {
        // Si es un valor simple
        serde_json::to_value($simple).unwrap()
    };
}

// Nota: Requiere la crate 'serde_json'
// let objeto = mi_json!({
//     nombre: "Maxi",
//     edad: 30,
//     direccion: { calle: "Falsa 123" }
// });

### B. El Patrón "Incremental Muncher"
Este patrón procesa la entrada token por token. Es útil para crear mini-lenguajes (DSLs) complejos.

In [ ]:
macro_rules! calcular {
    // Caso base: un solo número
    ($e:expr) => { $e };

    // Suma
    ($e:expr + $($resto:tt)*) => {
        $e + calcular!($($resto)*)
    };

    // Resta
    ($e:expr - $($resto:tt)*) => {
        $e - calcular!($($resto)*)
    };
}

let resultado = calcular!(10 + 5 - 2 + 1);
println!("Resultado: {}", resultado); // 14

## 2. Macros Procedimentales: El Siguiente Nivel

A diferencia de `macro_rules!`, las macros procedimentales son funciones escritas en Rust que operan sobre el flujo de tokens (`TokenStream`). 

### Estructura de un proyecto de macros procedimentales
Deben vivir en su propia crate con `proc-macro = true` en el `Cargo.toml`.

```toml
[lib]
proc-macro = true

[dependencies]
syn = "2.0"       # Para parsear tokens a estructuras de Rust
quote = "1.0"     # Para convertir estructuras de Rust de vuelta a tokens
```

### A. Custom Derive Macros
Permite que `#[derive(MiTrait)]` implemente automáticamente un trait para una estructura.

In [ ]:
/* 
Ejemplo de implementación en la crate de macros:

#[proc_macro_derive(MiLog)]
pub fn mi_log_derive(input: TokenStream) -> TokenStream {
    let ast = syn::parse(input).unwrap();
    impl_mi_log(&ast)
}

fn impl_mi_log(ast: &syn::DeriveInput) -> TokenStream {
    let name = &ast.ident;
    let gen = quote! {
        impl MiLog for #name {
            fn log(&self) {
                println!("Logueando instancia de: {}", stringify!(#name));
            }
        }
    };
    gen.into()
}
*/

### B. Attribute Macros
Crea atributos personalizados que pueden transformar funciones o structs enteros. Son la base de frameworks como `Rocket` o `Actix`.

In [ ]:
/*
#[route(GET, "/index")]
fn mi_manejador() {
    // El atributo 'route' podría registrar esta función en un servidor web
}
*/

## 3. Higiene y Visibilidad en Macros

- **Higiene en `macro_rules!`**: Rust asegura que las variables dentro de la macro no colisionen con las del exterior.
- **Higiene en Procedural Macros**: Es parcial. Se suele usar `crate::...` o `$crate` para referenciar tipos de la librería actual de forma segura.

### Ejemplo de colisión evitada (Higiene)
En otros lenguajes (como C), esto fallaría. En Rust, funciona perfectamente.

In [2]:
macro_rules! suma_con_cinco {
    ($val:expr) => {{
        let cinco = 5;
        $val + cinco
    }};
}

let cinco = 100;
let resultado = suma_con_cinco!(1); 
println!("Resultado: {}", resultado); // Imprime 6, no 101

Resultado: 6


## 4. Mejores Prácticas

1. **No abuses de las macros**: Si puedes usar una función o un trait, hazlo. Las macros son más difíciles de depurar.
2. **Usa `cargo-expand`**: Esta herramienta te permite ver qué código genera realmente tu macro. Es vital para debugging.
3. **Documentación**: Las macros suelen ser opacas. Documenta siempre los fragmentos que espera (`expr`, `ident`, `tt`, etc.).